# LeetCode 1656 — Design an Ordered Stream bằng Python

Link đề: https://leetcode.com/problems/design-an-ordered-stream/

Độ khó: **Easy** · Chủ đề: Array, Hash Table, Design, Data Stream

Skeleton LeetCode:

```python
class OrderedStream:

    def __init__(self, n: int):


    def insert(self, idKey: int, value: str) -> list[str]:



# Your OrderedStream object will be instantiated and called as such:
# obj = OrderedStream(n)
# param_1 = obj.insert(idKey,value)
```

Mục tiêu học:

1. Làm quen dạng bài **Design**: viết một `class` có trạng thái (state) và các method thay đổi trạng thái đó.
2. Hiểu `__init__`, `self`, thuộc tính đối tượng (`self.data`, `self.ptr`).
3. Từ cách "quét lại từ đầu mỗi lần" lên **con trỏ `ptr` chỉ tiến, không lùi**.
4. Hiểu phân tích **khấu hao** (amortized): một lần `insert` có thể tốn nhiều, nhưng **tổng** mọi lần chỉ `O(n)`.
5. Viết hàm chạy chuỗi lệnh kiểu LeetCode (`["OrderedStream","insert",...]`) để tự test.
6. Tự viết lại class và giải các biến thể stream.

## 1. Bài toán yêu cầu gì?

Có `n` cặp `(idKey, value)` đến theo **thứ tự bất kỳ**; `idKey` là số từ `1` đến `n`, không trùng nhau; `value` là chuỗi.

Thiết kế class `OrderedStream`:

- `OrderedStream(n)`: tạo stream nhận `n` giá trị.
- `insert(idKey, value)`: chèn cặp vào stream, rồi trả về **khối (chunk) dài nhất** các giá trị **tiếp theo theo thứ tự id** đã có sẵn.

Ghép tất cả các chunk trả về sẽ được danh sách value **đã sắp theo id**.

Ví dụ của đề (`n = 5`):

| Lệnh | Trả về | Vì sao |
|---|---|---|
| `insert(3, "ccccc")` | `[]` | đang chờ id 1, chưa có |
| `insert(1, "aaaaa")` | `["aaaaa"]` | có id 1; id 2 chưa có → dừng |
| `insert(2, "bbbbb")` | `["bbbbb", "ccccc"]` | có 2, và 3 đã có từ trước → lấy luôn |
| `insert(5, "eeeee")` | `[]` | đang chờ id 4 |
| `insert(4, "ddddd")` | `["ddddd", "eeeee"]` | có 4, 5 đã có |

Ghép: `[] + ["aaaaa"] + ["bbbbb","ccccc"] + [] + ["ddddd","eeeee"]` = theo đúng thứ tự id.

Constraints:

- `1 <= n <= 1000`, `1 <= id <= n`
- `value.length == 5`, chỉ chữ thường
- Mỗi lần `insert` có `id` khác nhau; có **đúng `n`** lần `insert`.

### Hình dung bằng "quầy trả hàng"

Tưởng tượng quầy gọi số: nhân viên luôn chờ **số kế tiếp** (`ptr`). Khách đến lộn xộn:

- Khách số 3 đến trước → ngồi chờ, quầy chưa gọi ai.
- Khách số 1 đến → gọi số 1; số 2 chưa đến → dừng.
- Khách số 2 đến → gọi 2, thấy 3 đang chờ → gọi luôn 3; số 4 chưa có → dừng.

Biến quan trọng nhất: **`ptr` = id nhỏ nhất chưa được trả ra**. `ptr` chỉ **tăng**.

In [1]:
arrivals = [(3, "ccccc"), (1, "aaaaa"), (2, "bbbbb"), (5, "eeeee"), (4, "ddddd")]
arrived = set()
ptr = 1
for id_key, value in arrivals:
    arrived.add(id_key)
    called = []
    while ptr in arrived:
        called.append(ptr)
        ptr += 1
    print(f"đến id {id_key}: gọi các số {called}, quầy giờ chờ số {ptr}")

đến id 3: gọi các số [], quầy giờ chờ số 1
đến id 1: gọi các số [1], quầy giờ chờ số 2
đến id 2: gọi các số [2, 3], quầy giờ chờ số 4
đến id 5: gọi các số [], quầy giờ chờ số 4
đến id 4: gọi các số [4, 5], quầy giờ chờ số 6


## 2. Nền tảng Python cần biết

### 2.1. `class`, `__init__` và `self`

- `class` là "khuôn" để tạo đối tượng có **dữ liệu** + **hành vi**.
- `__init__` chạy **một lần** khi tạo đối tượng: `obj = OrderedStream(5)` gọi `__init__(obj, 5)`.
- `self` là chính đối tượng đó. `self.ptr = 1` lưu dữ liệu **gắn với đối tượng**, các method khác đọc lại được.

In [2]:
class Counter2:
    def __init__(self, start: int):
        self.value = start          # thuộc tính của đối tượng

    def bump(self) -> int:
        self.value += 1             # đọc và sửa trạng thái
        return self.value

c1 = Counter2(10)
c2 = Counter2(100)
print(c1.bump(), c1.bump(), "| c2 độc lập:", c2.bump())
print("c1.value =", c1.value)

11 12 | c2 độc lập: 101
c1.value = 12


Mỗi đối tượng có **trạng thái riêng**: `c1` và `c2` không ảnh hưởng nhau. LeetCode sẽ tạo một `OrderedStream` mới cho mỗi test.

### 2.2. List có sẵn chỗ: `[None] * (n + 1)`

Ta cần "ô" cho mỗi id `1..n`. Dùng list dài `n + 1` và **bỏ trống ô 0** để `data[id]` khớp trực tiếp với id (không phải trừ 1). `None` nghĩa là "chưa có giá trị".

In [3]:
n = 5
data = [None] * (n + 1)
data[3] = "ccccc"
print(data)
print("ô 3 có dữ liệu?", data[3] is not None)
print("ô 1 có dữ liệu?", data[1] is not None)

[None, None, None, 'ccccc', None, None]
ô 3 có dữ liệu? True
ô 1 có dữ liệu? False


### 2.3. `while` dừng khi gặp ô trống — và kiểm tra biên trước

Vòng chính của bài:

```python
while self.ptr <= n and self.data[self.ptr] is not None:
```

Thứ tự hai điều kiện quan trọng: `and` **ngắt mạch**, nên khi `ptr > n` Python không đọc `data[ptr]` (tránh `IndexError`).

In [4]:
data = [None, "a", "b", "c"]
ptr = 1
while ptr <= 3 and data[ptr] is not None:
    print("lấy", data[ptr])
    ptr += 1
print("dừng với ptr =", ptr, "(đã vượt n = 3, không đọc data[4])")

lấy a
lấy b
lấy c
dừng với ptr = 4 (đã vượt n = 3, không đọc data[4])


### 2.4. Slicing để lấy cả khối

Nếu biết khối là các ô `start .. ptr-1`, có thể trả `data[start:ptr]` một lần thay vì `append` từng phần tử.

In [5]:
data = [None, "aaaaa", "bbbbb", "ccccc", None, None]
start, ptr = 2, 4
print(data[start:ptr])

['bbbbb', 'ccccc']


### 2.5. `getattr` để gọi method theo tên (dùng khi viết hàm test)

`getattr(obj, "insert")` trả về method `obj.insert`. Nhờ đó ta chạy được chuỗi lệnh dạng chữ của LeetCode.

In [6]:
c = Counter2(0)
method = getattr(c, "bump")
print(method(), method())

1 2


## 3. Cách trực giác — mỗi lần insert, quét lại từ id 1

Người mới hay nghĩ: lưu tất cả, rồi mỗi lần `insert` thì **duyệt từ id 1** để tìm chỗ đang chờ, và trả ra khối tiếp theo. Muốn biết "đã trả tới đâu" thì đánh dấu thêm mảng `sent`.

In [7]:
class OrderedStreamNaive:
    def __init__(self, n: int):
        self.n = n
        self.data = [None] * (n + 1)
        self.sent = [False] * (n + 1)

    def insert(self, idKey: int, value: str) -> list[str]:
        self.data[idKey] = value
        i = 1
        while i <= self.n and self.sent[i]:      # quét lại từ đầu
            i += 1
        chunk = []
        while i <= self.n and self.data[i] is not None:
            chunk.append(self.data[i])
            self.sent[i] = True
            i += 1
        return chunk

s = OrderedStreamNaive(5)
for id_key, value in [(3, "ccccc"), (1, "aaaaa"), (2, "bbbbb"), (5, "eeeee"), (4, "ddddd")]:
    print(f"insert({id_key}) ->", s.insert(id_key, value))

insert(3) -> []
insert(1) -> ['aaaaa']
insert(2) -> ['bbbbb', 'ccccc']
insert(5) -> []
insert(4) -> ['ddddd', 'eeeee']


### Nút thắt

Vòng `while ... self.sent[i]` **quét lại** các id đã gửi ở mỗi lần `insert`. Nếu id đến theo thứ tự `1, 2, 3, ..., n`, lần thứ `t` quét `t - 1` ô → tổng `0 + 1 + ... + (n-1) = O(n²)`.

Nhưng vị trí "đang chờ" **chỉ tiến lên, không bao giờ lùi**. Vậy sao không **nhớ** nó lại? Đó là con trỏ `ptr`.

In [8]:
def naive_scan_steps(n):
    return sum(t for t in range(n))   # lần thứ t quét t ô đã gửi

for n in [10, 100, 1000]:
    print(f"n = {n:>4}: quét lại ~ {naive_scan_steps(n):>7} ô")

n =   10: quét lại ~      45 ô
n =  100: quét lại ~    4950 ô
n = 1000: quét lại ~  499500 ô


## 4. Insight — con trỏ `ptr` chỉ tiến

Trạng thái cần nhớ:

- `data[id]`: value của id (hoặc `None`).
- `ptr`: id nhỏ nhất **chưa được trả ra**.

Mỗi `insert(idKey, value)`:

1. `data[idKey] = value`.
2. Nếu `data[ptr]` vẫn `None` (tức `idKey != ptr` và ô chờ chưa có) → trả `[]`.
3. Ngược lại, gom liên tục `data[ptr], data[ptr+1], ...` cho tới ô trống đầu tiên; `ptr` dừng ở ô trống đó.

Nhận xét: mỗi id được `ptr` "đi qua" **đúng một lần** trong cả vòng đời stream.

In [9]:
data = [None, None, None, "ccccc", None, None]
ptr = 1
print("ô chờ ptr=1 có dữ liệu?", data[ptr] is not None, "-> trả []")
data[1] = "aaaaa"
print("sau insert(1): ô chờ ptr=1 có dữ liệu?", data[ptr] is not None, "-> bắt đầu gom")

ô chờ ptr=1 có dữ liệu? False -> trả []
sau insert(1): ô chờ ptr=1 có dữ liệu? True -> bắt đầu gom


## 5. Lời giải chính — list + con trỏ

Giảng từng đoạn:

**`__init__`:** tạo `self.data = [None] * (n + 1)` và `self.ptr = 1`. Lưu `self.n = n` để kiểm tra biên.

**`insert` — đoạn 1:** ghi `self.data[idKey] = value`.

**`insert` — đoạn 2:** nhớ `start = self.ptr`, rồi `while self.ptr <= self.n and self.data[self.ptr] is not None: self.ptr += 1`.

**`insert` — đoạn 3:** trả `self.data[start:self.ptr]` (rỗng nếu `ptr` không tiến).

In [10]:
class OrderedStreamPointer:
    def __init__(self, n: int):
        self.n = n
        self.data = [None] * (n + 1)   # bỏ trống ô 0 để data[id] khớp id
        self.ptr = 1                   # id nhỏ nhất chưa trả ra

    def insert(self, idKey: int, value: str) -> list[str]:
        self.data[idKey] = value
        start = self.ptr
        while self.ptr <= self.n and self.data[self.ptr] is not None:
            self.ptr += 1
        return self.data[start:self.ptr]

s = OrderedStreamPointer(5)
for id_key, value in [(3, "ccccc"), (1, "aaaaa"), (2, "bbbbb"), (5, "eeeee"), (4, "ddddd")]:
    print(f"insert({id_key}, {value!r}) ->", s.insert(id_key, value))

insert(3, 'ccccc') -> []
insert(1, 'aaaaa') -> ['aaaaa']
insert(2, 'bbbbb') -> ['bbbbb', 'ccccc']
insert(5, 'eeeee') -> []
insert(4, 'ddddd') -> ['ddddd', 'eeeee']


`{value!r}` trong f-string in theo dạng `repr` (có dấu nháy), giúp phân biệt chuỗi với số khi debug.

## 6. TRACE từng bước

Bảng trạng thái ví dụ đề (`n = 5`). Cột `data[1..5]` dùng `·` cho ô trống.

| Lệnh | data[1..5] sau khi ghi | ptr trước | ptr sau | Trả về |
|---|---|---:|---:|---|
| insert(3,c) | · · c · · | 1 | 1 | [] |
| insert(1,a) | a · c · · | 1 | 2 | [a] |
| insert(2,b) | a b c · · | 2 | 4 | [b, c] |
| insert(5,e) | a b c · e | 4 | 4 | [] |
| insert(4,d) | a b c d e | 4 | 6 | [d, e] |

`ptr = 6 = n + 1` ở cuối: mọi giá trị đã được trả ra.

In [11]:
class OrderedStreamTrace(OrderedStreamPointer):
    def insert(self, idKey: int, value: str) -> list[str]:
        before = self.ptr
        result = super().insert(idKey, value)
        cells = " ".join(v[0] if v else "·" for v in self.data[1:])
        print(f"insert({idKey}, {value}) | data=[{cells}] | ptr {before} -> {self.ptr} | trả {result}")
        return result

t = OrderedStreamTrace(5)
for id_key, value in [(3, "ccccc"), (1, "aaaaa"), (2, "bbbbb"), (5, "eeeee"), (4, "ddddd")]:
    t.insert(id_key, value)

insert(3, ccccc) | data=[· · c · ·] | ptr 1 -> 1 | trả []
insert(1, aaaaa) | data=[a · c · ·] | ptr 1 -> 2 | trả ['aaaaa']
insert(2, bbbbb) | data=[a b c · ·] | ptr 2 -> 4 | trả ['bbbbb', 'ccccc']
insert(5, eeeee) | data=[a b c · e] | ptr 4 -> 4 | trả []
insert(4, ddddd) | data=[a b c d e] | ptr 4 -> 6 | trả ['ddddd', 'eeeee']


Hai cú pháp mới trong cell trace:

- `class OrderedStreamTrace(OrderedStreamPointer)`: **kế thừa** — lớp con dùng lại toàn bộ code của lớp cha.
- `super().insert(...)`: gọi method `insert` **của lớp cha**, rồi lớp con in thêm thông tin. Nhờ vậy trace chạy đúng code thật, không phải chép lại.

In [12]:
# trace một thứ tự khác: đến ngược hoàn toàn
t2 = OrderedStreamTrace(4)
for id_key in [4, 3, 2, 1]:
    t2.insert(id_key, "v" + str(id_key) + "xxx")

insert(4, v4xxx) | data=[· · · v] | ptr 1 -> 1 | trả []
insert(3, v3xxx) | data=[· · v v] | ptr 1 -> 1 | trả []
insert(2, v2xxx) | data=[· v v v] | ptr 1 -> 1 | trả []
insert(1, v1xxx) | data=[v v v v] | ptr 1 -> 5 | trả ['v1xxx', 'v2xxx', 'v3xxx', 'v4xxx']


## 7. Code nộp LeetCode

Tên class phải đúng `OrderedStream`, hai method đúng chữ ký `__init__(self, n)` và `insert(self, idKey, value)`.

In [13]:
class OrderedStream:

    def __init__(self, n: int):
        self.n = n
        self.data = [None] * (n + 1)
        self.ptr = 1

    def insert(self, idKey: int, value: str) -> list[str]:
        self.data[idKey] = value
        start = self.ptr
        while self.ptr <= self.n and self.data[self.ptr] is not None:
            self.ptr += 1
        return self.data[start:self.ptr]


obj = OrderedStream(5)
assert obj.insert(3, "ccccc") == []
assert obj.insert(1, "aaaaa") == ["aaaaa"]
assert obj.insert(2, "bbbbb") == ["bbbbb", "ccccc"]
assert obj.insert(5, "eeeee") == []
assert obj.insert(4, "ddddd") == ["ddddd", "eeeee"]
print("Ví dụ của đề PASS")

Ví dụ của đề PASS


### Hàm chạy chuỗi lệnh kiểu LeetCode

LeetCode mô tả test bằng hai list: tên lệnh và tham số. Ta viết hàm `run_ops` để tái hiện y hệt định dạng `Output` của đề (`None` thay cho `null`).

In [14]:
def run_ops(cls, ops: list[str], args: list[list]):
    obj = cls(*args[0])           # lệnh đầu là constructor
    out = [None]
    for name, a in zip(ops[1:], args[1:]):
        out.append(getattr(obj, name)(*a))
    return out

ops = ["OrderedStream", "insert", "insert", "insert", "insert", "insert"]
args = [[5], [3, "ccccc"], [1, "aaaaa"], [2, "bbbbb"], [5, "eeeee"], [4, "ddddd"]]
print(run_ops(OrderedStream, ops, args))

[None, [], ['aaaaa'], ['bbbbb', 'ccccc'], [], ['ddddd', 'eeeee']]


`cls(*args[0])`: dấu `*` **trải** list thành các đối số riêng, `f(*[5])` giống `f(5)`, `f(*[3, "c"])` giống `f(3, "c")`.

## 8. Các lời giải khác

### 8.1. Dùng `dict` thay cho list

Không cần biết trước kích thước ô; `ptr in self.data` kiểm tra id đã có chưa. Hợp khi id thưa hoặc không giới hạn.

In [15]:
class OrderedStreamDict:
    def __init__(self, n: int):
        self.data = {}
        self.ptr = 1

    def insert(self, idKey: int, value: str) -> list[str]:
        self.data[idKey] = value
        chunk = []
        while self.ptr in self.data:
            chunk.append(self.data[self.ptr])
            self.ptr += 1
        return chunk

print(run_ops(OrderedStreamDict, ops, args))

[None, [], ['aaaaa'], ['bbbbb', 'ccccc'], [], ['ddddd', 'eeeee']]


Cách dict không cần kiểm tra `ptr <= n` vì khi `ptr = n + 1`, key đó không bao giờ có trong dict.

### 8.2. Dọn bộ nhớ: xóa phần tử đã trả ra

Với stream rất dài, ta có thể `pop` các id đã trả để dict không phình to. `dict.pop(key)` vừa lấy giá trị vừa xóa key.

In [16]:
class OrderedStreamPop:
    def __init__(self, n: int):
        self.pending = {}
        self.ptr = 1

    def insert(self, idKey: int, value: str) -> list[str]:
        self.pending[idKey] = value
        chunk = []
        while self.ptr in self.pending:
            chunk.append(self.pending.pop(self.ptr))
            self.ptr += 1
        return chunk

s = OrderedStreamPop(5)
for id_key, value in [(3, "ccccc"), (1, "aaaaa"), (2, "bbbbb")]:
    print(s.insert(id_key, value), "| còn chờ:", s.pending)

[] | còn chờ: {3: 'ccccc'}
['aaaaa'] | còn chờ: {3: 'ccccc'}
['bbbbb', 'ccccc'] | còn chờ: {}


### 8.3. Bảng so sánh các cách

| Cách | Trạng thái | 1 lần insert (xấu nhất) | Tổng n lần insert | Bộ nhớ | Nhận xét |
|---|---|---|---|---|---|
| Quét lại từ đầu | list + `sent` | `O(n)` | `O(n²)` | `O(n)` | không nhớ vị trí chờ |
| List + `ptr` | list + `ptr` | `O(n)` | `O(n)` | `O(n)` | **nên dùng** |
| Dict + `ptr` | dict + `ptr` | `O(n)` | `O(n)` trung bình | `O(n)` | id thưa vẫn ổn |
| Dict + `pop` | dict chỉ giữ phần chờ | `O(n)` | `O(n)` trung bình | `O(số đang chờ)` | tiết kiệm bộ nhớ cho stream dài |

## 9. Vì sao lời giải đúng?

**Invariant** (đúng sau mỗi lần `insert`):

1. Mọi id `< ptr` **đã được trả ra**, theo đúng thứ tự tăng dần, mỗi id đúng một lần.
2. Ô `data[ptr]` **trống** (hoặc `ptr = n + 1`).

Chứng minh:

- Ban đầu `ptr = 1`, chưa trả gì — đúng.
- Khi `insert(idKey, value)`:
  - Nếu `idKey != ptr`, ô `data[ptr]` vẫn trống, vòng `while` không chạy, trả `[]`. Invariant giữ. Trả `[]` là đúng vì id kế tiếp chưa có.
  - Nếu `idKey == ptr`, vòng `while` gom `ptr, ptr+1, ...` tới ô trống đầu tiên. Đây chính là **khối dài nhất** có thể trả (không thể trả qua ô trống vì sẽ sai thứ tự). Sau vòng, `ptr` ở ô trống — invariant giữ.

Vì có đúng `n` lần insert với id khác nhau, cuối cùng mọi ô đều có dữ liệu, `ptr = n + 1`, và ghép các chunk = toàn bộ value theo id.

## 10. Độ phức tạp — phân tích khấu hao

Gọi `n` là số phần tử.

- `__init__`: `O(n)` thời gian và bộ nhớ (tạo list).
- Một lần `insert`: xấu nhất `O(n)` — ví dụ insert id `1` sau cùng khi `2..n` đã có, phải trả cả `n` phần tử.
- **Tổng `n` lần insert: `O(n)`**. Lý do: `ptr` chỉ tăng, từ `1` lên tối đa `n + 1`, nên tổng số lần `ptr += 1` qua mọi lời gọi `<= n`. Phần còn lại mỗi lần `insert` là `O(1)` (cộng thêm việc tạo chunk, có tổng độ dài cũng `n`).
- Vậy **khấu hao (amortized) `O(1)` mỗi lần insert**.
- Bộ nhớ: `O(n)`.

In [17]:
class OrderedStreamCount(OrderedStreamPointer):
    moves = 0
    def insert(self, idKey, value):
        before = self.ptr
        r = super().insert(idKey, value)
        OrderedStreamCount.moves += self.ptr - before
        return r

import random
rng = random.Random(1656)
for n in [10, 100, 1000]:
    OrderedStreamCount.moves = 0
    s = OrderedStreamCount(n)
    order = list(range(1, n + 1))
    rng.shuffle(order)
    for i in order:
        s.insert(i, "xxxxx")
    print(f"n = {n:>4}: tổng số bước ptr = {OrderedStreamCount.moves}")

n =   10: tổng số bước ptr = 10
n =  100: tổng số bước ptr = 100
n = 1000: tổng số bước ptr = 1000


Tổng số bước của `ptr` luôn đúng bằng `n` dù thứ tự đến thế nào — minh họa trực tiếp cho phân tích khấu hao.

## 11. Bộ test

1. Ví dụ đề (qua `run_ops`).
2. `n = 1`.
3. Đến theo thứ tự tăng: mỗi lần trả đúng 1 phần tử.
4. Đến theo thứ tự giảm: chỉ lần cuối trả cả khối.
5. Tính chất: ghép mọi chunk = list value sắp theo id (kiểm tra với nhiều hoán vị ngẫu nhiên).
6. Đối chiếu 4 cài đặt.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS -> {actual}")

check("đề", run_ops(OrderedStream, ops, args),
      [None, [], ["aaaaa"], ["bbbbb", "ccccc"], [], ["ddddd", "eeeee"]])
check("n = 1", run_ops(OrderedStream, ["OrderedStream", "insert"], [[1], [1, "zzzzz"]]),
      [None, ["zzzzz"]])
check("thứ tự tăng", run_ops(OrderedStream, ["OrderedStream"] + ["insert"] * 3,
      [[3], [1, "a"], [2, "b"], [3, "c"]]), [None, ["a"], ["b"], ["c"]])
check("thứ tự giảm", run_ops(OrderedStream, ["OrderedStream"] + ["insert"] * 3,
      [[3], [3, "c"], [2, "b"], [1, "a"]]), [None, [], [], ["a", "b", "c"]])

đề                     PASS -> [None, [], ['aaaaa'], ['bbbbb', 'ccccc'], [], ['ddddd', 'eeeee']]
n = 1                  PASS -> [None, ['zzzzz']]
thứ tự tăng            PASS -> [None, ['a'], ['b'], ['c']]
thứ tự giảm            PASS -> [None, [], [], ['a', 'b', 'c']]


In [19]:
impls = [OrderedStream, OrderedStreamNaive, OrderedStreamDict, OrderedStreamPop]
for _ in range(300):
    n = rng.randint(1, 15)
    order = list(range(1, n + 1))
    rng.shuffle(order)
    values = {i: "".join(rng.choice("abcde") for _ in range(5)) for i in order}
    outputs = []
    for cls in impls:
        s = cls(n)
        outputs.append([s.insert(i, values[i]) for i in order])
    assert all(o == outputs[0] for o in outputs), order
    flat = [v for chunk in outputs[0] for v in chunk]
    assert flat == [values[i] for i in range(1, n + 1)]
print("4 cài đặt khớp nhau; ghép các chunk luôn bằng list sắp theo id (300 hoán vị)")

4 cài đặt khớp nhau; ghép các chunk luôn bằng list sắp theo id (300 hoán vị)


## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng biến toàn cục / thuộc tính lớp thay vì `self`

Nếu `data` và `ptr` là biến của **lớp** (khai báo ngoài `__init__`), mọi đối tượng **dùng chung** — test sau bị dính dữ liệu test trước.

In [20]:
class WrongShared:
    data = {}      # thuộc tính LỚP, dùng chung!
    ptr = 1

    def __init__(self, n):
        pass

    def insert(self, idKey, value):
        self.data[idKey] = value
        chunk = []
        while WrongShared.ptr in self.data:
            chunk.append(self.data[WrongShared.ptr])
            WrongShared.ptr += 1
        return chunk

a = WrongShared(2); a.insert(1, "x"); a.insert(2, "y")
b = WrongShared(2)
print("stream mới b.insert(1) ->", b.insert(1, "p"), "(sai, phải là ['p'])")
print("stream đúng ->", OrderedStream(2).insert(1, "p"))

stream mới b.insert(1) -> [] (sai, phải là ['p'])
stream đúng -> ['p']


Sửa: luôn khởi tạo trạng thái trong `__init__` bằng `self.xxx = ...`.

### Lỗi 2: Quên kiểm tra `ptr <= n` khi dùng list

Khi đã trả hết, `ptr = n + 1` và `data[n + 1]` vượt chỉ số.

In [21]:
class WrongNoBound:
    def __init__(self, n):
        self.data = [None] * (n + 1)
        self.ptr = 1
    def insert(self, idKey, value):
        self.data[idKey] = value
        start = self.ptr
        while self.data[self.ptr] is not None:     # thiếu ptr <= n
            self.ptr += 1
        return self.data[start:self.ptr]

w = WrongNoBound(2)
w.insert(2, "b")
try:
    w.insert(1, "a")
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


### Lỗi 3: Chỉ trả về đúng phần tử vừa chèn

Sai: `return [value] if idKey == self.ptr else []` rồi `ptr += 1` — bỏ quên các id đã chờ sẵn phía sau (như `"ccccc"` khi chèn id 2).

In [22]:
class WrongSingle:
    def __init__(self, n):
        self.data = {}
        self.ptr = 1
    def insert(self, idKey, value):
        self.data[idKey] = value
        if idKey == self.ptr:
            self.ptr += 1
            return [value]
        return []

print("sai :", run_ops(WrongSingle, ops, args))
print("đúng:", run_ops(OrderedStream, ops, args))

sai : [None, [], ['aaaaa'], ['bbbbb'], [], []]
đúng: [None, [], ['aaaaa'], ['bbbbb', 'ccccc'], [], ['ddddd', 'eeeee']]


### Lỗi 4: Dùng chỉ số 0-based mà quên trừ 1

Nếu tạo `data = [None] * n` (ô `0..n-1`) thì phải ghi `data[idKey - 1]` và con trỏ bắt đầu từ `0`. Trộn hai quy ước là lỗi lệch 1 kinh điển. Mẹo của notebook: tạo `n + 1` ô, bỏ trống ô 0, để id dùng thẳng làm chỉ số.

## 13. Mẹo nhận dạng pattern

- Bài **Design** → xác định rõ **trạng thái** cần lưu (ở đây: `data`, `ptr`) rồi mới viết method.
- "Trả ra theo thứ tự khi phần tiếp theo sẵn sàng" → **con trỏ chỉ tiến** + bảng đánh dấu.
- Một thao tác có thể tốn `O(n)` nhưng con trỏ không lùi → **phân tích khấu hao**: tổng `O(n)`.
- id nằm trong `1..n` → list `n + 1` ô; id thưa / lớn → dict.

Ý tưởng tương tự xuất hiện trong: TCP reassembly (ghép gói tin đến lộn xộn), 1429 First Unique Number, 352 Data Stream as Disjoint Intervals, 2424 Longest Uploaded Prefix (gần như cùng bài).

## 14. Biến thể

### Biến thể 1: Longest Uploaded Prefix (LeetCode 2424)

Thay vì trả chunk, hỏi "tiền tố `1..x` đã upload dài nhất là bao nhiêu?". Chính là `ptr - 1`.

In [23]:
class LUPrefix:
    def __init__(self, n: int):
        self.seen = [False] * (n + 2)
        self.ptr = 1
    def upload(self, video: int) -> None:
        self.seen[video] = True
        while self.seen[self.ptr]:
            self.ptr += 1
    def longest(self) -> int:
        return self.ptr - 1

p = LUPrefix(4)
p.upload(3); print(p.longest())
p.upload(1); print(p.longest())
p.upload(2); print(p.longest())

0
1
3


Mẹo `n + 2` ô: ô `n + 1` luôn `False`, đóng vai "lính canh" (sentinel) để vòng `while` tự dừng mà không cần `ptr <= n`.

### Biến thể 2: Stream không biết trước n

Dùng `OrderedStreamDict` — không cần `n`. Thử chèn id lớn:

In [24]:
s = OrderedStreamDict(0)
print(s.insert(2, "b"), s.insert(1, "a"), s.insert(10, "z"), s.insert(3, "c"))

[] ['a', 'b'] [] ['c']


### Biến thể 3: Trả về chunk dạng chuỗi ghép

In [25]:
class OrderedStreamJoined(OrderedStream):
    def insert_joined(self, idKey: int, value: str) -> str:
        return "".join(self.insert(idKey, value))

s = OrderedStreamJoined(3)
print(repr(s.insert_joined(2, "bb")), repr(s.insert_joined(1, "aa")), repr(s.insert_joined(3, "cc")))

'' 'aabb' 'cc'


## 15. Bài tập tự luyện

Các cell TODO chỉ định nghĩa class/hàm, không được gọi khi chạy notebook.

### Bài 1 — Trace bằng tay

`n = 6`, thứ tự insert id: `2, 4, 1, 3, 6, 5`. Ghi `ptr` trước/sau và chunk trả về (dùng tên ngắn: id `i` có value `"v_i"`).

### Bài 2 — Class bộ đếm có trạng thái

Viết class `Ticket` với `__init__(self)` và `next(self) -> int` trả `1, 2, 3, ...` mỗi lần gọi.

In [26]:
class ExerciseTicket:
    def __init__(self):
        # TODO: khởi tạo trạng thái
        pass

    def next(self) -> int:
        # TODO: tăng và trả về
        pass

# t = ExerciseTicket(); assert [t.next(), t.next(), t.next()] == [1, 2, 3]

### Bài 3 — Viết lại `OrderedStream` bằng list + con trỏ

In [27]:
class ExerciseOrderedStream:
    def __init__(self, n: int):
        # TODO: data dài n + 1, ptr = 1
        pass

    def insert(self, idKey: int, value: str) -> list[str]:
        # TODO: ghi, nhớ start, tiến ptr, trả slice
        pass

# assert run_ops(ExerciseOrderedStream, ops, args) == [None, [], ["aaaaa"], ["bbbbb", "ccccc"], [], ["ddddd", "eeeee"]]

### Bài 4 — Viết lại bằng dict + `pop`

In [28]:
class ExerciseOrderedStreamPop:
    def __init__(self, n: int):
        # TODO
        pass

    def insert(self, idKey: int, value: str) -> list[str]:
        # TODO: while self.ptr in pending: pop
        pass

# assert run_ops(ExerciseOrderedStreamPop, ops, args) == [None, [], ["aaaaa"], ["bbbbb", "ccccc"], [], ["ddddd", "eeeee"]]

### Bài 5 — Hàm thuần: chunks từ một thứ tự đến

Không dùng class. Cho `n` và `order` (hoán vị của `1..n`), trả về list độ dài các chunk.

In [29]:
def exercise_chunk_sizes(n: int, order: list[int]) -> list[int]:
    # TODO: mô phỏng ptr bằng set hoặc list bool
    pass

# assert exercise_chunk_sizes(5, [3, 1, 2, 5, 4]) == [0, 1, 2, 0, 2]

### Bài 6 — Mini challenge: số lần insert trả chunk rỗng

Chứng minh (hoặc kiểm chứng bằng code) rằng số chunk **không rỗng** bằng số `i` trong `order` mà `i` xuất hiện **sau** tất cả `1..i-1`. Viết hàm đếm số chunk không rỗng **không cần mô phỏng stream**, chỉ dùng vị trí `pos[i]` của mỗi id trong `order`.

In [30]:
def exercise_nonempty_chunks(n: int, order: list[int]) -> int:
    # TODO: pos[i] = vị trí của i; đếm các i mà pos[i] > max(pos[1..i-1])
    pass

# assert exercise_nonempty_chunks(5, [3, 1, 2, 5, 4]) == 3

## 16. Gợi ý 3 mức

**Bài 1**
- Mức 1: chunk rỗng khi id chèn khác `ptr`.
- Mức 2: insert 1 khi 2 đã có → trả `[v1, v2]`, ptr thành 3.
- Mức 3: dãy ptr: `1,1,3,5,5,7`; chunk: `[]`, `[]`, `[v1,v2]`, `[v3,v4]`, `[]`, `[v5,v6]`.

**Bài 2**
- Mức 1: cần một thuộc tính lưu số hiện tại.
- Mức 2: `self.cur = 0` trong `__init__`.
- Mức 3: `self.cur += 1; return self.cur`.

**Bài 3**
- Mức 1: chép khung từ mục 5 mà không nhìn.
- Mức 2: điều kiện `while self.ptr <= self.n and self.data[self.ptr] is not None`.
- Mức 3: `return self.data[start:self.ptr]`.

**Bài 4**
- Mức 1: `self.pending = {}`.
- Mức 2: `while self.ptr in self.pending`.
- Mức 3: `chunk.append(self.pending.pop(self.ptr)); self.ptr += 1`.

**Bài 5**
- Mức 1: `arrived = [False] * (n + 2)`.
- Mức 2: sau mỗi id, tiến `ptr` khi `arrived[ptr]`.
- Mức 3: độ dài chunk = `ptr_sau - ptr_trước`.

**Bài 6**
- Mức 1: chunk không rỗng xảy ra đúng khi chèn id `== ptr`.
- Mức 2: id `i` được chèn khi `ptr == i` ⇔ mọi id `< i` đã đến trước `i`.
- Mức 3: duyệt `i = 1..n`, giữ `running_max = max(pos[1..i-1])`, đếm khi `pos[i] > running_max`.

## 17. Đáp án tham khảo

**Bài 1:** chunk lần lượt `[]`, `[]`, `["v_1","v_2"]`, `["v_3","v_4"]`, `[]`, `["v_5","v_6"]`; ptr sau mỗi lần: `1, 1, 3, 5, 5, 7`.

In [31]:
class AnswerTicket:
    def __init__(self):
        self.cur = 0
    def next(self):
        self.cur += 1
        return self.cur

class AnswerOrderedStream:
    def __init__(self, n):
        self.n, self.data, self.ptr = n, [None] * (n + 1), 1
    def insert(self, idKey, value):
        self.data[idKey] = value
        start = self.ptr
        while self.ptr <= self.n and self.data[self.ptr] is not None:
            self.ptr += 1
        return self.data[start:self.ptr]

class AnswerOrderedStreamPop:
    def __init__(self, n):
        self.pending, self.ptr = {}, 1
    def insert(self, idKey, value):
        self.pending[idKey] = value
        chunk = []
        while self.ptr in self.pending:
            chunk.append(self.pending.pop(self.ptr))
            self.ptr += 1
        return chunk

def answer_chunk_sizes(n, order):
    arrived = [False] * (n + 2)
    ptr, sizes = 1, []
    for i in order:
        arrived[i] = True
        before = ptr
        while arrived[ptr]:
            ptr += 1
        sizes.append(ptr - before)
    return sizes

def answer_nonempty_chunks(n, order):
    pos = [0] * (n + 1)
    for idx, i in enumerate(order):
        pos[i] = idx
    count, running_max = 0, -1
    for i in range(1, n + 1):
        if pos[i] > running_max:
            count += 1
        running_max = max(running_max, pos[i])
    return count

s = AnswerOrderedStream(6)
got = [s.insert(i, f"v_{i}") for i in [2, 4, 1, 3, 6, 5]]
assert got == [[], [], ["v_1", "v_2"], ["v_3", "v_4"], [], ["v_5", "v_6"]]   # Bài 1
t = AnswerTicket(); assert [t.next(), t.next(), t.next()] == [1, 2, 3]
expected = [None, [], ["aaaaa"], ["bbbbb", "ccccc"], [], ["ddddd", "eeeee"]]
assert run_ops(AnswerOrderedStream, ops, args) == expected
assert run_ops(AnswerOrderedStreamPop, ops, args) == expected
assert answer_chunk_sizes(5, [3, 1, 2, 5, 4]) == [0, 1, 2, 0, 2]
assert answer_nonempty_chunks(5, [3, 1, 2, 5, 4]) == 3
for _ in range(300):
    n = rng.randint(1, 12)
    order = list(range(1, n + 1)); rng.shuffle(order)
    sizes = answer_chunk_sizes(n, order)
    assert sum(sizes) == n
    assert sum(1 for x in sizes if x > 0) == answer_nonempty_chunks(n, order)
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class OrderedStream:

    def __init__(self, n: int):
        self.n = n
        self.data = [None] * (n + 1)   # ô 0 bỏ trống, data[id]
        self.ptr = 1                   # id nhỏ nhất chưa trả

    def insert(self, idKey: int, value: str) -> list[str]:
        self.data[idKey] = value
        start = self.ptr
        while self.ptr <= self.n and self.data[self.ptr] is not None:
            self.ptr += 1              # con trỏ chỉ tiến
        return self.data[start:self.ptr]
```

Câu thần chú: **"Lưu vào ô, con trỏ chờ ở ô trống đầu tiên, có thì tiến hết cỡ, trả đoạn vừa đi qua."**

Checklist trước khi submit:

- [ ] Tên class `OrderedStream`, method `insert(self, idKey, value)` đúng chữ ký?
- [ ] Trạng thái khởi tạo trong `__init__` bằng `self.` (không dùng thuộc tính lớp)?
- [ ] List dài `n + 1` (hoặc dùng dict) và kiểm tra `ptr <= n` trước khi đọc?
- [ ] Trả cả khối liên tiếp, không chỉ phần tử vừa chèn?
- [ ] Trả `[]` (list rỗng) khi id kế tiếp chưa có?
- [ ] Hiểu vì sao tổng chi phí là `O(n)` (khấu hao)?